# CUAD Contract Intelligence — Data Exploration & Cleaning
**Dataset:** CUAD v1 (Contract Understanding Atticus Dataset)  
**Goal:** Load, explore, clean, and save the master clauses CSV for downstream NLP tasks.

## 1. Environment Sanity Check

In [110]:
import sys
print("Python executable:", sys.executable)
print("Python version   :", sys.version)


Python executable: c:\Users\ATHARVA\OneDrive\Desktop\git\contract-intelligence\venv\Scripts\python.exe
Python version   : 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]


## 2. Install Dependencies

In [111]:
import sys

packages = [
    "datasets",
    "transformers",
    "spacy",
    "torch",
    "pandas",
    "numpy",
    "scikit-learn",
]

import subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
    check=True
)
print("✅ All packages installed / verified.")

✅ All packages installed / verified.


## 3. Import Libraries

In [112]:
import os
import re
import ast
import zipfile
import urllib.request

import pandas as pd
import numpy as np

print("✅ Libraries imported.")

✅ Libraries imported.


## 4. Download & Extract CUAD v1 Dataset
> Skip this cell if you already have the data locally.

In [113]:
DATA_DIR = "../data/raw"
ZIP_PATH = os.path.join(DATA_DIR, "CUAD_v1.zip")
URL      = "https://zenodo.org/record/4595826/files/CUAD_v1.zip"

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading CUAD_v1.zip ...")
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print("Download complete.")
else:
    print("Zip already exists — skipping download.")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

print("✅ Extracted. Contents:", os.listdir(DATA_DIR))


Zip already exists — skipping download.
✅ Extracted. Contents: ['CUAD_v1', 'CUAD_v1.zip']


## 5. List All Extracted Files

In [114]:
for root, dirs, files in os.walk("../data/raw"):
    for file in files:
        print(os.path.join(root, file))

../data/raw\CUAD_v1.zip
../data/raw\CUAD_v1\CUAD_v1.json
../data/raw\CUAD_v1\CUAD_v1_README.txt
../data/raw\CUAD_v1\master_clauses.csv
../data/raw\CUAD_v1\master_clauses_cleaned.csv
../data/raw\CUAD_v1\master_clauses_final.csv
../data/raw\CUAD_v1\full_contract_pdf\Part_I\Affiliate_Agreements\CreditcardscomInc_20070810_S-1_EX-10.33_362297_EX-10.33_Affiliate Agreement.pdf
../data/raw\CUAD_v1\full_contract_pdf\Part_I\Affiliate_Agreements\CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf
../data/raw\CUAD_v1\full_contract_pdf\Part_I\Affiliate_Agreements\DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate Agreement.pdf
../data/raw\CUAD_v1\full_contract_pdf\Part_I\Affiliate_Agreements\LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.pdf
../data/raw\CUAD_v1\full_contract_pdf\Part_I\Affiliate_Agreements\SouthernStarEnergyInc_20051202_SB-2A_EX-9_801890_EX-9_Affiliate Agreement.pdf
../data/raw\CUAD_v1\full_contract_pdf\P

## 6. Define File Paths

In [115]:
CSV_INPUT  = "../data/raw/CUAD_v1/master_clauses.csv"
CSV_OUTPUT = "../data/raw/CUAD_v1/master_clauses_final.csv"

print("Input  :", CSV_INPUT)
print("Output :", CSV_OUTPUT)


Input  : ../data/raw/CUAD_v1/master_clauses.csv
Output : ../data/raw/CUAD_v1/master_clauses_final.csv


## 7. Load the Dataset

In [116]:
data = pd.read_csv(CSV_INPUT, encoding="utf-8-sig")

print(f"Shape  : {data.shape}")
print(f"Columns: {data.columns.tolist()}")
data.head(3)


Shape  : (510, 83)
Columns: ['Filename', 'Document Name', 'Document Name-Answer', 'Parties', 'Parties-Answer', 'Agreement Date', 'Agreement Date-Answer', 'Effective Date', 'Effective Date-Answer', 'Expiration Date', 'Expiration Date-Answer', 'Renewal Term', 'Renewal Term-Answer', 'Notice Period To Terminate Renewal', 'Notice Period To Terminate Renewal- Answer', 'Governing Law', 'Governing Law-Answer', 'Most Favored Nation', 'Most Favored Nation-Answer', 'Competitive Restriction Exception', 'Competitive Restriction Exception-Answer', 'Non-Compete', 'Non-Compete-Answer', 'Exclusivity', 'Exclusivity-Answer', 'No-Solicit Of Customers', 'No-Solicit Of Customers-Answer', 'No-Solicit Of Employees', 'No-Solicit Of Employees-Answer', 'Non-Disparagement', 'Non-Disparagement-Answer', 'Termination For Convenience', 'Termination For Convenience-Answer', 'Rofr/Rofo/Rofn', 'Rofr/Rofo/Rofn-Answer', 'Change Of Control', 'Change Of Control-Answer', 'Anti-Assignment', 'Anti-Assignment-Answer', 'Revenue/

,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,...,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,['MARKETING AFFILIATE AGREEMENT'],MARKETING AFFILIATE AGREEMENT,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...","Birch First Global Investments Inc. (""Company""...","['8th day of May 2014', 'May 8, 2014']",5/8/14,['This agreement shall begin upon the date of ...,NaN,['This agreement shall begin upon the date of ...,...,[],No,"[""COMPANY'S SOLE AND EXCLUSIVE LIABILITY FOR T...",Yes,[],No,[],No,[],No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT'],VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"['EuroMedia Holdings Corp.', 'Rogers', 'Rogers...","Rogers Cable Communications Inc. (""Rogers""); E...","['July 11 , 2006']",7/11/06,"['July 11 , 2006']",7/11/06,"['The term of this Agreement (the ""Initial Ter...",...,[],No,[],No,[],No,[],No,[],No
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...,['CONTENT DISTRIBUTION AND LICENSE AGREEMENT'],CONTENT DISTRIBUTION AND LICENSE AGREEMENT,"['Producer', 'Fulucai Productions Ltd.', 'Conv...","CONVERGTV, INC. (“ConvergTV”); Fulucai Product...","['November 15, 2012']",11/15/12,"['November 15, 2012']",11/15/12,[],...,[],No,[],No,[],No,[],No,[],No


## 8. Exploratory Data Analysis
### 8a — Basic Statistic

In [117]:
print("Shape  :", data.shape)
print("\nNull counts per column:")
print(data.isnull().sum())


Shape  : (510, 83)

Null counts per column:
Filename                          0
Document Name                     0
Document Name-Answer              0
Parties                           0
Parties-Answer                    1
                                 ..
Insurance-Answer                  0
Covenant Not To Sue               0
Covenant Not To Sue-Answer        0
Third Party Beneficiary           0
Third Party Beneficiary-Answer    0
Length: 83, dtype: int64


### 8b — Unique Values per Column

In [118]:
data.nunique()

Filename                          510
Document Name                     274
Document Name-Answer              285
Parties                           502
Parties-Answer                    499
                                 ... 
Insurance-Answer                    2
Covenant Not To Sue               101
Covenant Not To Sue-Answer          2
Third Party Beneficiary            33
Third Party Beneficiary-Answer      2
Length: 83, dtype: int64

### 8c — Data Types

In [119]:
data.dtypes

Filename                          str
Document Name                     str
Document Name-Answer              str
Parties                           str
Parties-Answer                    str
                                 ... 
Insurance-Answer                  str
Covenant Not To Sue               str
Covenant Not To Sue-Answer        str
Third Party Beneficiary           str
Third Party Beneficiary-Answer    str
Length: 83, dtype: object

### 8d — Sample Rows

In [121]:
data.head(5)

,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,...,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,['MARKETING AFFILIATE AGREEMENT'],MARKETING AFFILIATE AGREEMENT,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...","Birch First Global Investments Inc. (""Company""...","['8th day of May 2014', 'May 8, 2014']",5/8/14,['This agreement shall begin upon the date of ...,NaN,['This agreement shall begin upon the date of ...,...,[],No,"[""COMPANY'S SOLE AND EXCLUSIVE LIABILITY FOR T...",Yes,[],No,[],No,[],No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT'],VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"['EuroMedia Holdings Corp.', 'Rogers', 'Rogers...","Rogers Cable Communications Inc. (""Rogers""); E...","['July 11 , 2006']",7/11/06,"['July 11 , 2006']",7/11/06,"['The term of this Agreement (the ""Initial Ter...",...,[],No,[],No,[],No,[],No,[],No
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...,['CONTENT DISTRIBUTION AND LICENSE AGREEMENT'],CONTENT DISTRIBUTION AND LICENSE AGREEMENT,"['Producer', 'Fulucai Productions Ltd.', 'Conv...","CONVERGTV, INC. (“ConvergTV”); Fulucai Product...","['November 15, 2012']",11/15/12,"['November 15, 2012']",11/15/12,[],...,[],No,[],No,[],No,[],No,[],No
3,GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10...,['WEBSITE CONTENT LICENSE AGREEMENT'],WEBSITE CONTENT LICENSE AGREEMENT,"['PSiTech Corporation', 'Licensor', 'Licensee'...","PSiTech Corporation (""Licensor""); Empirical Ve...","['Feb 10, 2014']",2/10/14,"['Feb 10, 2014']",2/10/14,['The initial term of this Agreement commences...,...,[],No,[],No,[],No,[],No,[],No
4,IdeanomicsInc_20160330_10-K_EX-10.26_9512211_E...,['CONTENT LICENSE AGREEMENT'],CONTENT LICENSE AGREEMENT,"['YOU ON DEMAND HOLDINGS, INC.', 'Licensor', '...",Beijing Sun Seven Stars Culture Development Li...,"['December 21, 2015']",12/21/15,"['December 21, 2015']",12/21/15,"['The Term of this Agreement (the ""Term"") shal...",...,[],No,[],No,[],No,[],No,[],No


In [120]:
data.head(5)

,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,...,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,['MARKETING AFFILIATE AGREEMENT'],MARKETING AFFILIATE AGREEMENT,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...","Birch First Global Investments Inc. (""Company""...","['8th day of May 2014', 'May 8, 2014']",5/8/14,['This agreement shall begin upon the date of ...,NaN,['This agreement shall begin upon the date of ...,...,[],No,"[""COMPANY'S SOLE AND EXCLUSIVE LIABILITY FOR T...",Yes,[],No,[],No,[],No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT'],VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"['EuroMedia Holdings Corp.', 'Rogers', 'Rogers...","Rogers Cable Communications Inc. (""Rogers""); E...","['July 11 , 2006']",7/11/06,"['July 11 , 2006']",7/11/06,"['The term of this Agreement (the ""Initial Ter...",...,[],No,[],No,[],No,[],No,[],No
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...,['CONTENT DISTRIBUTION AND LICENSE AGREEMENT'],CONTENT DISTRIBUTION AND LICENSE AGREEMENT,"['Producer', 'Fulucai Productions Ltd.', 'Conv...","CONVERGTV, INC. (“ConvergTV”); Fulucai Product...","['November 15, 2012']",11/15/12,"['November 15, 2012']",11/15/12,[],...,[],No,[],No,[],No,[],No,[],No
3,GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10...,['WEBSITE CONTENT LICENSE AGREEMENT'],WEBSITE CONTENT LICENSE AGREEMENT,"['PSiTech Corporation', 'Licensor', 'Licensee'...","PSiTech Corporation (""Licensor""); Empirical Ve...","['Feb 10, 2014']",2/10/14,"['Feb 10, 2014']",2/10/14,['The initial term of this Agreement commences...,...,[],No,[],No,[],No,[],No,[],No
4,IdeanomicsInc_20160330_10-K_EX-10.26_9512211_E...,['CONTENT LICENSE AGREEMENT'],CONTENT LICENSE AGREEMENT,"['YOU ON DEMAND HOLDINGS, INC.', 'Licensor', '...",Beijing Sun Seven Stars Culture Development Li...,"['December 21, 2015']",12/21/15,"['December 21, 2015']",12/21/15,"['The Term of this Agreement (the ""Term"") shal...",...,[],No,[],No,[],No,[],No,[],No


## 9. Create Working Copy

In [122]:
df = data.copy()
print("Working copy created. Shape:", df.shape)
data.head()

Working copy created. Shape: (510, 83)


,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,...,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,['MARKETING AFFILIATE AGREEMENT'],MARKETING AFFILIATE AGREEMENT,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...","Birch First Global Investments Inc. (""Company""...","['8th day of May 2014', 'May 8, 2014']",5/8/14,['This agreement shall begin upon the date of ...,NaN,['This agreement shall begin upon the date of ...,...,[],No,"[""COMPANY'S SOLE AND EXCLUSIVE LIABILITY FOR T...",Yes,[],No,[],No,[],No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT'],VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"['EuroMedia Holdings Corp.', 'Rogers', 'Rogers...","Rogers Cable Communications Inc. (""Rogers""); E...","['July 11 , 2006']",7/11/06,"['July 11 , 2006']",7/11/06,"['The term of this Agreement (the ""Initial Ter...",...,[],No,[],No,[],No,[],No,[],No
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...,['CONTENT DISTRIBUTION AND LICENSE AGREEMENT'],CONTENT DISTRIBUTION AND LICENSE AGREEMENT,"['Producer', 'Fulucai Productions Ltd.', 'Conv...","CONVERGTV, INC. (“ConvergTV”); Fulucai Product...","['November 15, 2012']",11/15/12,"['November 15, 2012']",11/15/12,[],...,[],No,[],No,[],No,[],No,[],No
3,GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10...,['WEBSITE CONTENT LICENSE AGREEMENT'],WEBSITE CONTENT LICENSE AGREEMENT,"['PSiTech Corporation', 'Licensor', 'Licensee'...","PSiTech Corporation (""Licensor""); Empirical Ve...","['Feb 10, 2014']",2/10/14,"['Feb 10, 2014']",2/10/14,['The initial term of this Agreement commences...,...,[],No,[],No,[],No,[],No,[],No
4,IdeanomicsInc_20160330_10-K_EX-10.26_9512211_E...,['CONTENT LICENSE AGREEMENT'],CONTENT LICENSE AGREEMENT,"['YOU ON DEMAND HOLDINGS, INC.', 'Licensor', '...",Beijing Sun Seven Stars Culture Development Li...,"['December 21, 2015']",12/21/15,"['December 21, 2015']",12/21/15,"['The Term of this Agreement (the ""Term"") shal...",...,[],No,[],No,[],No,[],No,[],No


## 10. Define Cell Cleaning Function

In [123]:
def clean_cell(value):
    """
    Cleans a single cell value:
    - Returns empty string for nulls
    - Flattens list-like strings into plain text
    - Strips brackets, quotes, tabs, newlines
    - Collapses multiple spaces
    - Removes spaces before punctuation
    """
    # Handle nulls
    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Handle empty list strings
    if value in ["[]", "['']", '[""]']:
        return ""

    # Handle list-like strings e.g. "['item1', 'item2']"
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                cleaned_items = [
                    str(item).strip()
                    for item in parsed
                    if item is not None and str(item).strip()
                ]
                value = " ".join(cleaned_items)
        except (ValueError, SyntaxError):
            value = value.strip("[]")

    # Strip brackets, quotes
    value = value.replace("[", "").replace("]", "")
    value = value.replace('"', "").replace("'", "")

    # Remove tabs and newlines
    value = re.sub(r"[\r\n\t]+", " ", value)

    # Collapse multiple spaces
    value = re.sub(r"\s+", " ", value)

    # Remove spaces before punctuation
    value = re.sub(r"\s+([,.!?;:])", r"\1", value)

    return value.strip()

print("✅ clean_cell() function defined.")


✅ clean_cell() function defined.


## 11. Apply Cleaning to All Columns

In [124]:
for col in df.columns:
    df[col] = df[col].apply(clean_cell)

print(f"✅ Cleaned all {len(df.columns)} columns.")
df.head(3)

✅ Cleaned all 83 columns.


,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,...,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,MARKETING AFFILIATE AGREEMENT,MARKETING AFFILIATE AGREEMENT,BIRCH FIRST GLOBAL INVESTMENTS INC. MA Marketi...,Birch First Global Investments Inc. (Company);...,"8th day of May 2014 May 8, 2014",5/8/14,This agreement shall begin upon the date of it...,,This agreement shall begin upon the date of it...,...,,No,COMPANYS SOLE AND EXCLUSIVE LIABILITY FOR THE ...,Yes,,No,,No,,No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,EuroMedia Holdings Corp. Rogers Rogers Cable C...,Rogers Cable Communications Inc. (Rogers); Eur...,"July 11, 2006",7/11/06,"July 11, 2006",7/11/06,The term of this Agreement (the Initial Term) ...,...,,No,,No,,No,,No,,No
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...,CONTENT DISTRIBUTION AND LICENSE AGREEMENT,CONTENT DISTRIBUTION AND LICENSE AGREEMENT,Producer Fulucai Productions Ltd. ConvergTV CO...,"CONVERGTV, INC. (“ConvergTV”); Fulucai Product...","November 15, 2012",11/15/12,"November 15, 2012",11/15/12,,...,,No,,No,,No,,No,,No


In [131]:
def is_valid_date(val):
    if pd.isna(val) or str(val).strip() == "":
        return False
    
    val = str(val).strip()
    
    # Remove if it's ONLY symbols, blanks, commas, dots
    if re.fullmatch(r'[_\-•·●,\.\s\*]+', val):
        return False
    
    # Remove if it has 2+ consecutive underscores (unfilled blank)
    if re.search(r'_{2,}', val):
        return False
    
    # Remove if it has no actual month name or numeric date
    # Must contain either a month name OR a valid number pattern
    has_month  = bool(re.search(
        r'\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\w*\b',
        val, re.IGNORECASE
    ))
    has_number = bool(re.search(r'\d{1,2}[\-\/]\d{1,2}', val))  # e.g. 03/24
    has_year   = bool(re.search(r'\b(19|20)\d{2}\b', val))       # e.g. 2015

    # Must have at least a month + year OR numeric date
    if not ((has_month and has_year) or has_number):
        return False
    
    return True

# ── Apply to Agreement Date-Answer column ─────────────────
col = "Agreement Date-Answer"

before = df[col].notna().sum()
df[col] = df[col].apply(lambda x: x if is_valid_date(x) else "")
after  = (df[col] != "").sum()

print(f"'{col}'")
print(f"  Before: {before} non-empty values")
print(f"  After : {after} valid dates")
print(f"  Removed: {before - after} blank/template entries")

'Agreement Date-Answer'
  Before: 510 non-empty values
  After : 420 valid dates
  Removed: 90 blank/template entries


## 12. Drop Fully Empty Columns & Rows

In [132]:
# Replace blank strings with NA for accurate detection
df = df.replace("", pd.NA)

# Drop columns where ALL values are empty
before = df.shape
df = df.dropna(axis=1, how="all")
print(f"Columns: {before[1]} → {df.shape[1]}  (removed {before[1] - df.shape[1]} empty columns)")

# Drop rows where ALL values are empty
df = df.dropna(axis=0, how="all")
print(f"Rows   : {before[0]} → {df.shape[0]}  (removed {before[0] - df.shape[0]} empty rows)")

# Fill remaining NAs with empty string
df = df.fillna("")

print(f"\n✅ Final shape: {df.shape}")


Columns: 83 → 83  (removed 0 empty columns)
Rows   : 510 → 510  (removed 0 empty rows)

✅ Final shape: (510, 83)


## 13. Drop Duplicate Rows

In [133]:
before = len(df)
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print(f"Rows before: {before} | After: {len(df)} | Removed: {before - len(df)} duplicates")


Rows before: 510 | After: 510 | Removed: 0 duplicates


## 14. Reorder Columns
Put identification columns first, then each clause paired with its Answer column.

In [134]:
columns  = list(df.columns)
new_order = []

# Step 1: Identification columns first
for col in ["Filename", "Document Name"]:
    if col in columns:
        new_order.append(col)

# Step 2: Pair each clause column with its -Answer column
for col in columns:
    if col in new_order:
        continue
    if str(col).lower().endswith("-answer"):
        continue  # skip; will be added after its parent

    new_order.append(col)
    answer_col = col + "-Answer"
    if answer_col in columns:
        new_order.append(answer_col)

# Step 3: Add anything remaining not yet in new_order
for col in columns:
    if col not in new_order:
        new_order.append(col)

df = df[new_order]
print(f"✅ Columns reordered. Final column count: {len(df.columns)}")
print(df.columns.tolist())


✅ Columns reordered. Final column count: 83
['Filename', 'Document Name', 'Parties', 'Parties-Answer', 'Agreement Date', 'Agreement Date-Answer', 'Effective Date', 'Effective Date-Answer', 'Expiration Date', 'Expiration Date-Answer', 'Renewal Term', 'Renewal Term-Answer', 'Notice Period To Terminate Renewal', 'Notice Period To Terminate Renewal- Answer', 'Governing Law', 'Governing Law-Answer', 'Most Favored Nation', 'Most Favored Nation-Answer', 'Competitive Restriction Exception', 'Competitive Restriction Exception-Answer', 'Non-Compete', 'Non-Compete-Answer', 'Exclusivity', 'Exclusivity-Answer', 'No-Solicit Of Customers', 'No-Solicit Of Customers-Answer', 'No-Solicit Of Employees', 'No-Solicit Of Employees-Answer', 'Non-Disparagement', 'Non-Disparagement-Answer', 'Termination For Convenience', 'Termination For Convenience-Answer', 'Rofr/Rofo/Rofn', 'Rofr/Rofo/Rofn-Answer', 'Change Of Control', 'Change Of Control-Answer', 'Anti-Assignment', 'Anti-Assignment-Answer', 'Revenue/Profit S

## 15. Final Row Filter
Remove any rows that are completely empty across all columns.

In [135]:
before = len(df)

df = df[
    df.apply(
        lambda row: any(str(x).strip() != "" for x in row),
        axis=1
    )
]

df = df.reset_index(drop=True)
print(f"Rows before: {before} | After: {len(df)} | Removed: {before - len(df)}")


Rows before: 510 | After: 510 | Removed: 0


## 16. Final Preview

In [136]:
print("Final shape:", df.shape)
print("\nSample data:")
df.head(5)


Final shape: (510, 83)

Sample data:


,Filename,Document Name,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,Expiration Date-Answer,...,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer,Document Name-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,MARKETING AFFILIATE AGREEMENT,BIRCH FIRST GLOBAL INVESTMENTS INC. MA Marketi...,Birch First Global Investments Inc. (Company);...,"8th day of May 2014 May 8, 2014",5/8/14,This agreement shall begin upon the date of it...,,This agreement shall begin upon the date of it...,12/31/14,...,No,COMPANYS SOLE AND EXCLUSIVE LIABILITY FOR THE ...,Yes,,No,,No,,No,MARKETING AFFILIATE AGREEMENT
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,EuroMedia Holdings Corp. Rogers Rogers Cable C...,Rogers Cable Communications Inc. (Rogers); Eur...,"July 11, 2006",7/11/06,"July 11, 2006",7/11/06,The term of this Agreement (the Initial Term) ...,6/30/10,...,No,,No,,No,,No,,No,VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...,CONTENT DISTRIBUTION AND LICENSE AGREEMENT,Producer Fulucai Productions Ltd. ConvergTV CO...,"CONVERGTV, INC. (“ConvergTV”); Fulucai Product...","November 15, 2012",11/15/12,"November 15, 2012",11/15/12,,,...,No,,No,,No,,No,,No,CONTENT DISTRIBUTION AND LICENSE AGREEMENT
3,GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10...,WEBSITE CONTENT LICENSE AGREEMENT,PSiTech Corporation Licensor Licensee Empirica...,PSiTech Corporation (Licensor); Empirical Vent...,"Feb 10, 2014",2/10/14,"Feb 10, 2014",2/10/14,The initial term of this Agreement commences a...,2/10/19,...,No,,No,,No,,No,,No,WEBSITE CONTENT LICENSE AGREEMENT
4,IdeanomicsInc_20160330_10-K_EX-10.26_9512211_E...,CONTENT LICENSE AGREEMENT,"YOU ON DEMAND HOLDINGS, INC. Licensor Licensee...",Beijing Sun Seven Stars Culture Development Li...,"December 21, 2015",12/21/15,"December 21, 2015",12/21/15,The Term of this Agreement (the Term) shall co...,12/21/35,...,No,,No,,No,,No,,No,CONTENT LICENSE AGREEMENT


## 17. Save Cleaned File

In [137]:
import tempfile

# Try project output path first, fall back to temp dir
try:
    output_path = CSV_OUTPUT
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
except PermissionError:
    output_path = os.path.join(tempfile.gettempdir(), "master_clauses_final.csv")
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("⚠️  Permission denied on project path — saved to temp folder instead.")

print(f"✅ File saved successfully!")
print(f"   Path  : {output_path}")
print(f"   Shape : {df.shape}")


⚠️  Permission denied on project path — saved to temp folder instead.
✅ File saved successfully!
   Path  : C:\Users\ATHARVA\AppData\Local\Temp\master_clauses_final.csv
   Shape : (510, 83)
